## EDA with SQL

Load the cleaned launch data into an in-memory SQL table and answer questions with queries instead of charts.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('dataset_part_2.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', conn, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql(sql, conn)

### Unique launch sites

In [2]:
q("""SELECT DISTINCT LaunchSite FROM SPACEXTBL;""")

,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


### 5 records where launch site begins with 'CCA'

In [3]:
q("""SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5;""")

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0


### Total payload mass carried by all boosters

In [4]:
q("""SELECT SUM(PayloadMass) AS TotalPayloadMass FROM SPACEXTBL;""")

,TotalPayloadMass
0,549446.347059


### Average payload mass by booster Block (F9 v1.1-era booster-version detail
was dropped during cleaning, so Block generation is the closest available
equivalent to the original 'F9 v1.1' breakdown)

In [5]:
q("""SELECT Block, AVG(PayloadMass) AS AvgPayloadMass FROM SPACEXTBL GROUP BY Block ORDER BY Block;""")

,Block,AvgPayloadMass
0,1.0,2578.839969
1,2.0,3848.166667
2,3.0,5320.863961
3,4.0,4900.073583
4,5.0,8811.426124


### Date of the first successful ground-pad landing

In [6]:
q("""SELECT MIN(Date) AS FirstSuccessfulGroundLanding FROM SPACEXTBL WHERE Outcome = 'True RTLS';""")

,FirstSuccessfulGroundLanding
0,2015-12-22


### Boosters with a successful drone-ship landing, payload 4,000-6,000 kg

In [7]:
q("""SELECT DISTINCT Serial FROM SPACEXTBL WHERE Outcome = 'True ASDS' AND PayloadMass > 4000 AND PayloadMass < 6000;""")

,Serial
0,B1022
1,B1026
2,B1021
3,B1031
4,B1046
5,B1059


### Total successful vs. failed mission outcomes

In [8]:
q("""SELECT SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS TotalSuccesses,
       SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS TotalFailures
FROM SPACEXTBL;""")

,TotalSuccesses,TotalFailures
0,60,30


### Booster(s) that carried the maximum payload

In [9]:
q("""SELECT Serial, PayloadMass FROM SPACEXTBL WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL);""")

,Serial,PayloadMass
0,B1048,15600.0
1,B1051,15600.0
2,B1048,15600.0


### 2015 drone-ship landing failures

In [10]:
q("""SELECT Outcome, BoosterVersion, LaunchSite, Date FROM SPACEXTBL
WHERE Outcome LIKE 'False%' AND strftime('%Y', Date) = '2015';""")

,Outcome,BoosterVersion,LaunchSite,Date
0,False ASDS,Falcon 9,CCAFS SLC 40,2015-01-10
1,False ASDS,Falcon 9,CCAFS SLC 40,2015-04-14


### Landing outcomes ranked by count, 2010-06-04 to 2017-03-20

In [11]:
q("""SELECT Outcome, COUNT(*) AS OutcomeCount FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Outcome ORDER BY OutcomeCount DESC;""")

,Outcome,OutcomeCount
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2
